# LeetCode 2104 — Sum of Subarray Ranges bằng Python

Link đề: https://leetcode.com/problems/sum-of-subarray-ranges/ · Độ khó: **Medium** · Chủ đề: Array, Stack, Monotonic Stack

Skeleton LeetCode:

```python
class Solution:
    def subArrayRanges(self, nums: List[int]) -> int:
        ...
```

Lộ trình:
1. **Brute force** tính lại `max − min` cho từng mảng con — `O(n³)`.
2. **Mở rộng đầu phải, giữ max/min chạy** — `O(n²)` (đủ qua với `n ≤ 1000`).
3. **Đóng góp bằng ngăn xếp đơn điệu**: tổng range = `Σ max(mảng con) − Σ min(mảng con)`; mỗi phần tử là max của bao nhiêu mảng con? — `O(n)` (follow-up, **nộp bài**).
4. **DP + ngăn xếp đơn điệu** theo đầu phải: `S_max[i] = S_max[p] + a[i]·(i − p)` — `O(n)`, góc nhìn khác của cùng ý tưởng.

## 1. Mục tiêu học

1. Tách bài thành **hai bài con độc lập**: tổng max của mọi mảng con và tổng min của mọi mảng con (tính tuyến tính của tổng).
2. Đếm số mảng con nhận `a[i]` làm max bằng **phần tử lớn hơn gần nhất bên trái/phải** — dùng ngăn xếp đơn điệu.
3. Xử lý **phần tử bằng nhau** đúng cách: một phía so sánh ngặt, phía kia không ngặt, để mỗi mảng con được gán cho **đúng một** phần tử.
4. Viết phiên bản DP theo đầu phải và hiểu vì sao hai cách cho cùng kết quả.
5. Kiểm chứng trên mảng nhiều giá trị trùng (nơi lỗi xử lý bằng nhau lộ ra).

## 2. Đề bài & ví dụ

**Range** của một mảng con = phần tử lớn nhất − phần tử nhỏ nhất. Trả về tổng range của **mọi** mảng con (liên tiếp, không rỗng).

**Ví dụ 1:** `[1,2,3]` → `4` (`0+0+0+1+1+2`).

**Ví dụ 2:** `[1,3,3]` → `4` (`0+0+0+2+0+2`).

**Ví dụ 3:** `[4,-2,-3,4,1]` → `59`.

**Ràng buộc:** `1 ≤ n ≤ 1000`, `-10⁹ ≤ nums[i] ≤ 10⁹`. **Follow-up:** `O(n)`.

In [1]:
import random
import time
from typing import List

examples = [([1, 2, 3], 4), ([1, 3, 3], 4), ([4, -2, -3, 4, 1], 59)]
for nums, want in examples:
    print(nums, "->", want)

[1, 2, 3] -> 4
[1, 3, 3] -> 4
[4, -2, -3, 4, 1] -> 59


## 3. Kiến thức nền

### 3.1 Tuyến tính của tổng

`Σ (max − min) = Σ max − Σ min`. Hai tổng tính riêng.

### 3.2 Phần tử lớn hơn gần nhất bên trái bằng ngăn xếp đơn điệu

Duyệt trái → phải, giữ ngăn xếp chỉ số có giá trị **giảm dần**. Trước khi đẩy `i`, bỏ mọi phần tử `≤ a[i]`; đỉnh còn lại (nếu có) là phần tử **lớn hơn** gần nhất bên trái.

In [2]:
def prev_greater(a):
    res, st = [-1] * len(a), []
    for i, x in enumerate(a):
        while st and a[st[-1]] <= x:
            st.pop()
        res[i] = st[-1] if st else -1
        st.append(i)
    return res

a = [4, -2, -3, 4, 1]
print("prev_greater (chỉ số):", prev_greater(a))

prev_greater (chỉ số): [-1, 0, 1, -1, 3]


Mỗi chỉ số vào và ra ngăn xếp tối đa một lần ⇒ `O(n)`.

### 3.3 Đếm mảng con chứa vị trí `i` trong khoảng `(L, R)`

Nếu đầu trái chọn trong `(L, i]` và đầu phải trong `[i, R)` thì có `(i − L) · (R − i)` mảng con chứa `i`.

In [3]:
L, i, R = 1, 3, 6
subs = [(l, r) for l in range(L + 1, i + 1) for r in range(i, R)]
print(len(subs), "=", (i - L) * (R - i), subs)

6 = 6 [(2, 3), (2, 4), (2, 5), (3, 3), (3, 4), (3, 5)]


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1 `[1, 2, 3]`: liệt kê mọi mảng con, max và min:

| mảng con | max | min | range |
|---|---|---|---|
| [1] | 1 | 1 | 0 |
| [2] | 2 | 2 | 0 |
| [3] | 3 | 3 | 0 |
| [1,2] | 2 | 1 | 1 |
| [2,3] | 3 | 2 | 1 |
| [1,2,3] | 3 | 1 | 2 |

`Σ max = 1 + 2 + 3 + 2 + 3 + 3 = 14`, `Σ min = 1 + 2 + 3 + 1 + 2 + 1 = 10` ⇒ `14 − 10 = 4` ✓.

**Đếm theo phần tử:** `3` là max của 3 mảng con (`[3]`, `[2,3]`, `[1,2,3]`), `2` là max của 2, `1` là max của 1 ⇒ `3·3 + 2·2 + 1·1 = 14` ✓.

In [4]:
nums = [1, 2, 3]; n = len(nums)
from collections import Counter
cmax, cmin = Counter(), Counter()
for l in range(n):
    for r in range(l, n):
        seg = nums[l:r + 1]
        cmax[seg.index(max(seg)) + l] += 1; cmin[seg.index(min(seg)) + l] += 1
print("số mảng con nhận nums[i] làm max:", dict(cmax), "| làm min:", dict(cmin))
print("Σmax =", sum(nums[i] * c for i, c in cmax.items()), "Σmin =", sum(nums[i] * c for i, c in cmin.items()))

số mảng con nhận nums[i] làm max: {0: 1, 1: 2, 2: 3} | làm min: {0: 3, 1: 2, 2: 1}
Σmax = 14 Σmin = 10


## 5. Cách 1 — Brute Force / baseline: tính lại max/min cho mỗi mảng con

In [5]:
def ranges_brute(nums):
    n, total = len(nums), 0
    for l in range(n):
        for r in range(l, n):
            seg = nums[l:r + 1]
            total += max(seg) - min(seg)
    return total

for nums, want in examples:
    print(ranges_brute(nums), "kỳ vọng", want)

4 kỳ vọng 4
4 kỳ vọng 4
59 kỳ vọng 59


**Độ phức tạp:** `O(n²)` mảng con × `O(n)` tính max/min ⇒ `O(n³) ≈ 1.7·10⁸` với `n = 1000` — quá chậm trong Python. Space `O(n)` cho lát cắt.

**Nút thắt:** khi mở rộng đầu phải một ô, max/min mới chỉ phụ thuộc max/min cũ và phần tử mới.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (`O(n²)`):** cố định `l`, tăng `r`; giữ `mx = max(mx, a[r])`, `mn = min(mn, a[r])`.

**Insight 2 (`O(n)`, đóng góp):** `a[i]` là max của mảng con `[l, r]` (có `l ≤ i ≤ r`) khi mọi phần tử trong đó `≤ a[i]` — nhưng nếu có phần tử **bằng** `a[i]`, phải chọn **một** chỉ số làm "đại diện". Quy ước: đại diện là phần tử max **trái nhất**. Khi đó:
- bên trái `i` (trong mảng con) mọi phần tử `< a[i]` ⇒ `L = ` chỉ số gần nhất bên trái có `a[L] ≥ a[i]`;
- bên phải `i` mọi phần tử `≤ a[i]` ⇒ `R = ` chỉ số gần nhất bên phải có `a[R] > a[i]`;
- số mảng con: `(i − L) · (R − i)`.

`Σ max = Σ a[i] · (i − L_i) · (R_i − i)`; tương tự cho min với chiều so sánh ngược.

**Quy tắc bằng nhau:** một phía `≥` (dừng ở phần tử bằng), phía kia `>` (vượt qua phần tử bằng). Hai phía cùng ngặt ⇒ đếm trùng; cùng không ngặt ⇒ bỏ sót.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — `O(n²)` với max/min chạy

In [6]:
def ranges_n2(nums):
    n, total = len(nums), 0
    for l in range(n):
        mx = mn = nums[l]
        for r in range(l + 1, n):
            x = nums[r]
            if x > mx: mx = x
            elif x < mn: mn = x
            total += mx - mn
    return total

for nums, want in examples:
    print(ranges_n2(nums), "kỳ vọng", want)

4 kỳ vọng 4
4 kỳ vọng 4
59 kỳ vọng 59


**Độ phức tạp:** `O(n²) = 5·10⁵` vòng với `n = 1000` — chạy tốt. Space `O(1)`. (Mảng con độ dài 1 có range 0 nên bắt đầu `r` từ `l + 1`.)

### 7.2 Cách 3 — Đóng góp bằng ngăn xếp đơn điệu (code nộp)

Một lượt ngăn xếp cho mỗi tổng: khi phần tử `j` **bị bật** khỏi ngăn xếp bởi `i`, ta biết ngay `R_j = i` và `L_j` = phần tử dưới nó trong ngăn xếp.

In [7]:
def sum_of_maxima(a):
    n = len(a); total = 0; st = []           # ngăn xếp chỉ số, giá trị giảm dần (không ngặt)
    for i in range(n + 1):
        cur = a[i] if i < n else float("inf")   # lính canh: bật hết ở cuối
        while st and a[st[-1]] < cur:           # bật khi gặp phần tử LỚN HƠN NGẶT (R: > a[j])
            j = st.pop()
            L = st[-1] if st else -1            # phần tử còn lại >= a[j] (L: >= a[j])
            total += a[j] * (j - L) * (i - j)
        st.append(i)
    return total

def sum_of_minima(a):
    return -sum_of_maxima([-x for x in a])    # min(a) = -max(-a)

def ranges_stack(nums):
    return sum_of_maxima(nums) - sum_of_minima(nums)

for nums, want in examples:
    print(ranges_stack(nums), "kỳ vọng", want, "| Σmax =", sum_of_maxima(nums), "Σmin =", sum_of_minima(nums))

4 kỳ vọng 4 | Σmax = 14 Σmin = 10
4 kỳ vọng 4 | Σmax = 16 Σmin = 12
59 kỳ vọng 59 | Σmax = 38 Σmin = -21


Phân tích quy ước bằng nhau trong `sum_of_maxima`: `j` bị bật bởi phần tử **lớn hơn ngặt** ⇒ `R_j` = phần tử lớn hơn ngặt gần nhất bên phải. Phần tử nằm dưới `j` trong ngăn xếp có giá trị `≥ a[j]` (vì phần tử bằng không bật nhau) ⇒ `L_j` = phần tử `≥` gần nhất bên trái. Vậy trong các mảng con `(L_j, R_j)` chứa `j`: bên trái `j` mọi phần tử `< a[j]`, bên phải mọi phần tử `≤ a[j]` ⇒ `j` là max **trái nhất** — đúng quy ước ở mục 6.

Code nộp LeetCode của Cách 3: xem mục 9.

**Độ phức tạp:** `O(n)` thời gian (mỗi chỉ số vào/ra ngăn xếp một lần), `O(n)` bộ nhớ.

### 7.3 Cách 4 — DP theo đầu phải với ngăn xếp đơn điệu

Gọi `M[i]` = tổng max của mọi mảng con **kết thúc tại `i`**. Nếu `p` là chỉ số gần nhất bên trái có `a[p] > a[i]` (hoặc `−1`), thì mọi mảng con `[l, i]` với `l > p` có max `= a[i]` (`i − p` mảng con), còn với `l ≤ p` thì max bằng max của `[l, p]`... nối thêm phần sau không đổi max ⇒ đóng góp `M[p]`.

`M[i] = M[p] + a[i] · (i − p)`; `Σ max = Σ_i M[i]`.

In [8]:
def sum_max_dp(a):
    n = len(a); M = [0] * n; st = []; total = 0
    for i, x in enumerate(a):
        while st and a[st[-1]] <= x:
            st.pop()
        p = st[-1] if st else -1
        M[i] = (M[p] if p >= 0 else 0) + x * (i - p)
        total += M[i]
        st.append(i)
    return total

def ranges_dp(nums):
    return sum_max_dp(nums) + sum_max_dp([-x for x in nums])     # Σmax − Σmin = Σmax(a) + Σmax(−a)

for nums, want in examples:
    print(ranges_dp(nums), "kỳ vọng", want)

4 kỳ vọng 4
4 kỳ vọng 4
59 kỳ vọng 59


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ. Ưu điểm: cho luôn "tổng max các mảng con kết thúc tại mỗi `i`" — hữu ích cho truy vấn tiền tố.

## 8. TRACE từng bước

Trace Cách 3 (`sum_of_maxima`) trên ví dụ 3: mỗi lần bật, in `j`, `L`, `R = i`, số mảng con và đóng góp.

In [9]:
def trace_maxima(a):
    n = len(a); total = 0; st = []
    print(f"a = {a}")
    for i in range(n + 1):
        cur = a[i] if i < n else float("inf")
        while st and a[st[-1]] < cur:
            j = st.pop(); L = st[-1] if st else -1
            add = a[j] * (j - L) * (i - j); total += add
            print(f"  i = {i} ({'lính canh' if i == n else a[i]}) bật j = {j} (a = {a[j]}): L = {L}, R = {i}, "
                  f"số mảng con = {(j - L)}·{(i - j)} = {(j - L) * (i - j)}, đóng góp {add}, tổng {total}")
        st.append(i)
        if i < n: print(f"  đẩy {i}; ngăn xếp (giá trị) = {[a[k] for k in st]}")
    return total

print("Σmax =", trace_maxima([4, -2, -3, 4, 1]))

a = [4, -2, -3, 4, 1]
  đẩy 0; ngăn xếp (giá trị) = [4]
  đẩy 1; ngăn xếp (giá trị) = [4, -2]
  đẩy 2; ngăn xếp (giá trị) = [4, -2, -3]
  i = 3 (4) bật j = 2 (a = -3): L = 1, R = 3, số mảng con = 1·1 = 1, đóng góp -3, tổng -3
  i = 3 (4) bật j = 1 (a = -2): L = 0, R = 3, số mảng con = 1·2 = 2, đóng góp -4, tổng -7
  đẩy 3; ngăn xếp (giá trị) = [4, 4]
  đẩy 4; ngăn xếp (giá trị) = [4, 4, 1]
  i = 5 (lính canh) bật j = 4 (a = 1): L = 3, R = 5, số mảng con = 1·1 = 1, đóng góp 1, tổng -6
  i = 5 (lính canh) bật j = 3 (a = 4): L = 0, R = 5, số mảng con = 3·2 = 6, đóng góp 24, tổng 18
  i = 5 (lính canh) bật j = 0 (a = 4): L = -1, R = 5, số mảng con = 1·5 = 5, đóng góp 20, tổng 38
Σmax = 38


In [10]:
def trace_dp(a):
    st = []; M = [0] * len(a)
    print(f"{'i':>2} | a[i] | p  | M[p] | a[i]·(i−p) | M[i]")
    for i, x in enumerate(a):
        while st and a[st[-1]] <= x: st.pop()
        p = st[-1] if st else -1
        M[i] = (M[p] if p >= 0 else 0) + x * (i - p)
        print(f"{i:>2} | {x:>4} | {p:>2} | {M[p] if p >= 0 else 0:>4} | {x * (i - p):>10} | {M[i]}")
        st.append(i)
    print("Σmax =", sum(M))

trace_dp([4, -2, -3, 4, 1])

 i | a[i] | p  | M[p] | a[i]·(i−p) | M[i]
 0 |    4 | -1 |    0 |          4 | 4
 1 |   -2 |  0 |    4 |         -2 | 2
 2 |   -3 |  1 |    2 |         -3 | -1
 3 |    4 | -1 |    0 |         16 | 16
 4 |    1 |  3 |   16 |          1 | 17
Σmax = 38


Hai cách cho cùng `Σmax` (xem output); trừ `Σmin` tính tương tự cho ra `59` ✓.

## 9. Code nộp LeetCode

In [11]:
class Solution:
    def subArrayRanges(self, nums: List[int]) -> int:
        def sum_max(a):
            n = len(a)
            total = 0
            st = []
            for i in range(n + 1):
                cur = a[i] if i < n else float("inf")
                while st and a[st[-1]] < cur:
                    j = st.pop()
                    left = st[-1] if st else -1
                    total += a[j] * (j - left) * (i - j)
                st.append(i)
            return total

        return sum_max(nums) + sum_max([-x for x in nums])

sol = Solution()
for nums, want in examples:
    got = sol.subArrayRanges(nums)
    print(got, "OK" if got == want else "SAI")

4 OK
4 OK
59 OK


## 10. Vì sao thuật toán đúng

**Bổ đề (gán duy nhất).** Với mỗi mảng con `[l, r]`, gọi `j*` là chỉ số **trái nhất** trong `[l, r]` đạt giá trị max. Khi đó mọi phần tử trong `[l, j*)` **nhỏ hơn ngặt** `a[j*]` và mọi phần tử trong `(j*, r]` **`≤`** `a[j*]`. Ngược lại, với `j` cố định, đặt `L_j` = chỉ số gần nhất bên trái có `a[L_j] ≥ a[j]` và `R_j` = chỉ số gần nhất bên phải có `a[R_j] > a[j]`; các mảng con nhận `j` làm "max trái nhất" chính là các `[l, r]` với `L_j < l ≤ j ≤ r < R_j`. Mỗi mảng con có đúng một max trái nhất ⇒ được đếm đúng một lần. ✓

**Code khớp định nghĩa.** Trong `sum_of_maxima`, `j` chỉ bị bật bởi phần tử **lớn hơn ngặt** ⇒ lúc bật, `i = R_j`. Khi `j` được đẩy vào, mọi phần tử `< a[j]` đã bị bật nên phần tử ngay dưới `j` có giá trị `≥ a[j]` và là gần nhất như vậy ⇒ đó là `L_j`. ✓

(Quy ước "phải nhất" — `L` ngặt, `R` không ngặt — cũng đúng; điều bắt buộc là **một** phía ngặt, phía kia không ngặt.)

**Đếm.** Số mảng con chứa `j` trong `(L_j, R_j)`: `(j − L_j) · (R_j − j)` (mục 3.3). Cộng `a[j] · số đó` cho `Σmax`. `Σmin(a) = −Σmax(−a)`. ✓

**Cách 4.** Với `p` = phần tử `> a[i]` gần nhất bên trái: các mảng con `[l, i]`, `l ∈ (p, i]` có max `a[i]`; các mảng con `[l, i]`, `l ≤ p` có max = max của `[l, p]` (vì `a[p] > a[i] ≥` mọi phần tử trong `(p, i]`). Tổng của nhóm sau đúng bằng `M[p]`. ✓

## 11. Phân tích Time / Space

`n ≤ 1000` (follow-up muốn `O(n)`).

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(n³)` | `O(n)` |
| 2. Max/min chạy | `O(n²)` | `O(1)` |
| 3. Ngăn xếp đóng góp | `O(n)` | `O(n)` |
| 4. DP + ngăn xếp | `O(n)` | `O(n)` |

Đo với `n = 1000` (giới hạn đề) và `n = 10⁵` (chỉ cách `O(n)`):

In [12]:
rng = random.Random(1)
a1000 = [rng.randint(-10**9, 10**9) for _ in range(1000)]
a_dup = [rng.randint(0, 3) for _ in range(1000)]
for name, arr in [("ngẫu nhiên", a1000), ("nhiều trùng", a_dup)]:
    res = []
    for fname, fn in [("n2", ranges_n2), ("stack", ranges_stack), ("dp", ranges_dp), ("submit", Solution().subArrayRanges)]:
        t0 = time.perf_counter(); r = fn(arr); res.append(r)
        print(f"{name:11s} {fname:6s}: {r}  {(time.perf_counter() - t0) * 1000:.1f} ms")
    assert len(set(res)) == 1
big = [rng.randint(-10**9, 10**9) for _ in range(10**5)]
t0 = time.perf_counter(); r1 = ranges_stack(big); t1 = time.perf_counter(); r2 = ranges_dp(big); t2 = time.perf_counter()
assert r1 == r2
print(f"n = 1e5: stack {t1 - t0:.2f}s, dp {t2 - t1:.2f}s, kết quả có {len(str(r1))} chữ số (vượt int64 nếu n lớn hơn nữa)")

ngẫu nhiên  n2    : 979172540120473  33.7 ms
ngẫu nhiên  stack : 979172540120473  0.5 ms
ngẫu nhiên  dp    : 979172540120473  0.5 ms
ngẫu nhiên  submit: 979172540120473  0.5 ms
nhiều trùng n2    : 1492865  25.6 ms
nhiều trùng stack : 1492865  0.5 ms
nhiều trùng dp    : 1492865  0.5 ms
nhiều trùng submit: 1492865  0.5 ms


n = 1e5: stack 0.04s, dp 0.04s, kết quả có 19 chữ số (vượt int64 nếu n lớn hơn nữa)


## 12. Bộ test

Tập trung vào **giá trị trùng** (miền giá trị nhỏ) — nơi lỗi quy ước bằng nhau xuất hiện.

In [13]:
fns = [ranges_n2, ranges_stack, ranges_dp, Solution().subArrayRanges]

for nums, want in examples:
    assert ranges_brute(nums) == want
    for fn in fns:
        assert fn(nums) == want

edge = [([5], 0), ([2, 2], 0), ([7, 7, 7, 7], 0), ([1, 2], 1), ([2, 1], 1),
        ([3, 1, 3], 6), ([1, 3, 1, 3], 2 + 2 + 2 + 2 + 2 + 2), ([-10**9, 10**9], 2 * 10**9)]
for nums, want in edge:
    assert ranges_brute(nums) == want, nums
    for fn in fns:
        assert fn(nums) == want, (fn, nums)

rng = random.Random(2024)
for _ in range(2000):
    n = rng.randint(1, 25)
    nums = [rng.randint(-3, 3) if rng.random() < 0.7 else rng.randint(-10**9, 10**9) for _ in range(n)]
    want = ranges_brute(nums)
    for fn in fns:
        assert fn(nums) == want, (fn, nums)
print("ví dụ + edge + 2000 mảng ngẫu nhiên (nhiều giá trị trùng) khớp brute force")

ví dụ + edge + 2000 mảng ngẫu nhiên (nhiều giá trị trùng) khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Cả hai phía đều so sánh **ngặt** (đếm trùng khi có giá trị bằng nhau)

In [14]:
def wrong_both_strict(a):
    n = len(a); L = [-1] * n; R = [n] * n; st = []
    for i in range(n):                                    # L: phần tử > a[i] gần nhất bên trái
        while st and a[st[-1]] <= a[i]: st.pop()
        L[i] = st[-1] if st else -1; st.append(i)
    st = []
    for i in range(n - 1, -1, -1):                        # R: phần tử > a[i] gần nhất bên phải (SAI: cũng ngặt)
        while st and a[st[-1]] <= a[i]: st.pop()
        R[i] = st[-1] if st else n; st.append(i)
    smax = sum(a[i] * (i - L[i]) * (R[i] - i) for i in range(n))
    b = [-x for x in a]; Lb = [-1] * n; Rb = [n] * n; st = []
    for i in range(n):
        while st and b[st[-1]] <= b[i]: st.pop()
        Lb[i] = st[-1] if st else -1; st.append(i)
    st = []
    for i in range(n - 1, -1, -1):
        while st and b[st[-1]] <= b[i]: st.pop()
        Rb[i] = st[-1] if st else n; st.append(i)
    smin = -sum(b[i] * (i - Lb[i]) * (Rb[i] - i) for i in range(n))
    return smax - smin

for nums, want in examples + [([3, 1, 3], 6), ([2, 2, 1], 2)]:
    print(f"{nums}: sai = {wrong_both_strict(nums)} | đúng = {want}")

[1, 2, 3]: sai = 4 | đúng = 4
[1, 3, 3]: sai = 7 | đúng = 4
[4, -2, -3, 4, 1]: sai = 67 | đúng = 59
[3, 1, 3]: sai = 9 | đúng = 6
[2, 2, 1]: sai = 4 | đúng = 2


Với `[3, 1, 3]`: mảng con `[3, 1, 3]` có hai max bằng nhau, cả hai đều "nhận" nó ⇒ đếm max hai lần. Ví dụ 1 không có giá trị trùng nên không lộ lỗi. **Sửa:** một phía `≥`, phía kia `>`.

### Lỗi 2 — Cả hai phía đều **không ngặt** (bỏ sót)

In [15]:
def wrong_both_nonstrict(a):
    def smax(a):
        n = len(a); L = [-1] * n; R = [n] * n; st = []
        for i in range(n):
            while st and a[st[-1]] < a[i]: st.pop()          # L: >= a[i]
            L[i] = st[-1] if st else -1; st.append(i)
        st = []
        for i in range(n - 1, -1, -1):
            while st and a[st[-1]] < a[i]: st.pop()          # R: >= a[i]  (SAI: cũng không ngặt)
            R[i] = st[-1] if st else n; st.append(i)
        return sum(a[i] * (i - L[i]) * (R[i] - i) for i in range(n))
    return smax(a) + smax([-x for x in a])

for nums, want in examples + [([3, 1, 3], 6), ([2, 2, 1], 2)]:
    print(f"{nums}: sai = {wrong_both_nonstrict(nums)} | đúng = {want}")

[1, 2, 3]: sai = 4 | đúng = 4
[1, 3, 3]: sai = 1 | đúng = 4
[4, -2, -3, 4, 1]: sai = 51 | đúng = 59
[3, 1, 3]: sai = 3 | đúng = 6
[2, 2, 1]: sai = 0 | đúng = 2


Mảng con có hai max bằng nhau thì **không** phần tử nào nhận (mỗi cái bị chặn bởi cái kia). **Sửa:** như Lỗi 1.

### Lỗi 3 — Nhầm dấu khi trừ tổng min

In [16]:
def wrong_sign(nums):
    return sum_of_maxima(nums) + sum_of_minima(nums)          # SAI: range = max − min

for nums, want in examples:
    print(f"{nums}: sai = {wrong_sign(nums)} | đúng = {want}")

[1, 2, 3]: sai = 24 | đúng = 4
[1, 3, 3]: sai = 28 | đúng = 4
[4, -2, -3, 4, 1]: sai = 17 | đúng = 59


`sum_of_minima` đã trả về `Σmin` (dương/âm theo dữ liệu); phải **trừ**. Lỗi hay gặp khi dùng mẹo `Σmax(−a) = −Σmin(a)` rồi lẫn lộn dấu. **Sửa:** `Σmax − Σmin` hoặc `Σmax(a) + Σmax(−a)` — chọn một và kiểm tra với ví dụ.

### Lỗi 4 — Quên lính canh cuối mảng (các phần tử còn trong ngăn xếp không được tính)

In [17]:
def wrong_no_sentinel(a):
    def smax(a):
        total, st = 0, []
        for i in range(len(a)):                               # SAI: không bật phần còn lại ở cuối
            while st and a[st[-1]] < a[i]:
                j = st.pop(); L = st[-1] if st else -1
                total += a[j] * (j - L) * (i - j)
            st.append(i)
        return total
    return smax(a) + smax([-x for x in a])

for nums, want in examples:
    print(f"{nums}: sai = {wrong_no_sentinel(nums)} | đúng = {want}")

[1, 2, 3]: sai = 5 | đúng = 4
[1, 3, 3]: sai = 1 | đúng = 4
[4, -2, -3, 4, 1]: sai = -11 | đúng = 59


Các chỉ số còn trong ngăn xếp khi kết thúc có `R = n` (không có phần tử lớn hơn bên phải) — phải xử lý chúng. **Sửa:** chạy thêm `i = n` với giá trị `+∞` (như code nộp), hoặc vòng xử lý riêng sau vòng chính.

### Lỗi 5 — `O(n²)` nhưng cắt lát để lấy max/min (thực chất `O(n³)`)

`max(nums[l:r+1])` trong hai vòng lặp là `O(n³)`: với `n = 1000` mất cả phút. Cách 2 cập nhật max/min **tăng dần** mới thật sự `O(n²)`.

In [18]:
t0 = time.perf_counter(); ranges_brute(a1000[:300]); t1 = time.perf_counter(); ranges_n2(a1000[:300]); t2 = time.perf_counter()
print(f"n = 300: cắt lát {t1 - t0:.2f}s | max/min chạy {t2 - t1:.3f}s")

n = 300: cắt lát 0.08s | max/min chạy 0.002s


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tổng của max/min trên **mọi** mảng con" ⇒ **đóng góp từng phần tử** + ngăn xếp đơn điệu (phần tử lớn/nhỏ hơn gần nhất hai phía).
- Giá trị trùng ⇒ **một phía ngặt, một phía không ngặt**. Luôn test `[3, 1, 3]`, `[2, 2, 1]`.
- Mẹo `min(a) = −max(−a)` để viết một hàm dùng hai lần.
- Lính canh `+∞` ở cuối để bật hết ngăn xếp.
- Họ bài: 907 *Sum of Subarray Minimums*, 1856 *Maximum Subarray Min-Product*, 2281 *Sum of Total Strength of Wizards*, 84 *Largest Rectangle in Histogram*, 496/503 *Next Greater Element*.

## 15. Biến thể của bài

1. **907 Sum of Subarray Minimums** (modulo `10⁹ + 7`): đúng `sum_of_minima`.
2. **Tổng range các mảng con độ dài đúng `k`**: hai deque đơn điệu cho max/min cửa sổ trượt (Bài 5).
3. **Tổng max − min trên mọi dãy con** (không liên tiếp, LeetCode 891 *Sum of Subsequence Widths*): sắp xếp, phần tử thứ `i` là max của `2^i` dãy con và min của `2^(n−1−i)`.
4. **Đếm số mảng con có range `≤ t`**: hai con trỏ + deque (LeetCode 1438).

In [19]:
MOD = 10**9 + 7
def sum_subarray_mins_907(arr):
    """Biến thể 1."""
    return sum_of_minima(arr) % MOD

def sum_subsequence_widths_891(nums):
    """Biến thể 3."""
    a = sorted(nums); n = len(a); pw = [1] * n
    for i in range(1, n): pw[i] = pw[i - 1] * 2 % MOD
    return sum(a[i] * (pw[i] - pw[n - 1 - i]) for i in range(n)) % MOD

from itertools import combinations
rng = random.Random(15)
for _ in range(200):
    nums = [rng.randint(1, 9) for _ in range(rng.randint(1, 8))]
    brute = sum(max(c) - min(c) for k in range(1, len(nums) + 1) for c in combinations(nums, k)) % MOD
    assert sum_subsequence_widths_891(nums) == brute
print("907 ví dụ [3,1,2,4] ->", sum_subarray_mins_907([3, 1, 2, 4]), "(kỳ vọng 17)")
print("891 ví dụ [2,1,3] ->", sum_subsequence_widths_891([2, 1, 3]), "(kỳ vọng 6)")

907 ví dụ [3,1,2,4] -> 17 (kỳ vọng 17)
891 ví dụ [2,1,3] -> 6 (kỳ vọng 6)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | tính lại | `O(n³)` | `O(n)` | kiểm chứng |
| 2. Max/min chạy | mở rộng đầu phải | `O(n²)` | `O(1)` | `n ≤ ~3000`, dễ viết |
| 3. Ngăn xếp đóng góp | đếm mảng con mỗi phần tử làm max/min | `O(n)` | `O(n)` | **nộp bài**, follow-up |
| 4. DP + ngăn xếp | tổng theo đầu phải | `O(n)` | `O(n)` | cần giá trị theo tiền tố |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `[5, 5]`? `[1, 4, 2]`? `[0, 10, 0]`?

**Bài 2 (tính tay):** với `[3, 1, 3]`, tính số mảng con nhận mỗi chỉ số làm max theo quy ước của code nộp (`L`: `≥`, `R`: `>`), rồi `Σmax`.

**Bài 3 (điền TODO):** hoàn thiện `next_greater_or_n(a)` trả về mảng `R` với `R[i]` = chỉ số gần nhất bên phải có `a[R[i]] > a[i]`, hoặc `n`.

**Bài 4 (tự viết lại):** viết `max_contrib(a)` trả về list `cnt` với `cnt[i]` = số mảng con nhận `i` làm max (theo quy ước "max trái nhất"); kiểm tra `Σ cnt = n(n+1)/2`.

**Bài 5 (mini challenge):** tổng range của mọi mảng con **độ dài đúng `k`**, `O(n)` bằng hai deque đơn điệu. Kiểm chứng bằng brute force. (Biến thể tự đặt — đối chiếu lại đề gốc nếu dùng làm đề.)

In [20]:
def next_greater_or_n(a):
    # TODO: duyệt từ phải sang trái, ngăn xếp giảm dần, bỏ các phần tử <= a[i]
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) `[5,5]`: mọi range 0. (2) `[1,4,2]`: `[1,4]=3`, `[4,2]=2`, `[1,4,2]=3` ⇒ `8`. (3) `[0,10,0]`: `10 + 10 + 10 = 30`.

**Bài 2:** (1) Chỉ số 0: không có phần tử `≥ 3` bên trái ⇒ `L = −1`; không có phần tử `> 3` bên phải ⇒ `R = 3` ⇒ `1 · 3 = 3` mảng con. (2) Chỉ số 1: `L = 0`, `R = 2` ⇒ `1`; chỉ số 2: `L = 0` (vì `a[0] = 3 ≥ 3`), `R = 3` ⇒ `2 · 1 = 2`. Tổng `3 + 1 + 2 = 6 = 3·4/2` ✓. (3) `Σmax = 3·3 + 1·1 + 3·2 = 16`.

**Bài 3:** (1) `R = [n] * n`, `st = []`. (2) `while st and a[st[-1]] <= a[i]: st.pop()`. (3) `R[i] = st[-1] if st else n`, rồi đẩy `i`.

**Bài 4:** (1) `L` = phần tử `≥` gần nhất bên trái, `R` = phần tử `>` gần nhất bên phải. (2) `cnt[i] = (i − L) · (R − i)`. (3) Đối chiếu đếm trực tiếp.

**Bài 5:** (1) Deque chỉ số giảm dần cho max, tăng dần cho min. (2) Đẩy `i`, bỏ chỉ số ra khỏi cửa sổ ở đầu deque. (3) Khi `i ≥ k − 1` cộng `a[dmax[0]] − a[dmin[0]]`.

## 19. Đáp án tham khảo

In [21]:
# Bài 1
assert [Solution().subArrayRanges(a) for a in ([5, 5], [1, 4, 2], [0, 10, 0])] == [0, 8, 30]
# Bài 2
assert sum_of_maxima([3, 1, 3]) == 16 and trace_maxima([3, 1, 3]) == 16

# Bài 3
def next_greater_or_n_ans(a):
    n = len(a); R = [n] * n; st = []
    for i in range(n - 1, -1, -1):
        while st and a[st[-1]] <= a[i]: st.pop()
        R[i] = st[-1] if st else n; st.append(i)
    return R

# Bài 4
def max_contrib_ans(a):
    n = len(a); L = [-1] * n; st = []
    for i in range(n):
        while st and a[st[-1]] < a[i]: st.pop()                  # còn lại >= a[i]
        L[i] = st[-1] if st else -1; st.append(i)
    R = next_greater_or_n_ans(a)                                 # > a[i]
    return [(i - L[i]) * (R[i] - i) for i in range(n)]

rng = random.Random(41)
for _ in range(500):
    a = [rng.randint(0, 4) for _ in range(rng.randint(1, 15))]; n = len(a)
    assert next_greater_or_n_ans(a) == [next((j for j in range(i + 1, n) if a[j] > a[i]), n) for i in range(n)]
    cnt = max_contrib_ans(a); direct = [0] * n
    for l in range(n):
        for r in range(l, n):
            seg = a[l:r + 1]; direct[l + seg.index(max(seg))] += 1     # max trái nhất
    assert cnt == direct and sum(cnt) == n * (n + 1) // 2
print("Bài 4 ví dụ [3,1,3]:", max_contrib_ans([3, 1, 3]))
print("Bài 1-4: OK")

a = [3, 1, 3]
  đẩy 0; ngăn xếp (giá trị) = [3]
  đẩy 1; ngăn xếp (giá trị) = [3, 1]
  i = 2 (3) bật j = 1 (a = 1): L = 0, R = 2, số mảng con = 1·1 = 1, đóng góp 1, tổng 1
  đẩy 2; ngăn xếp (giá trị) = [3, 3]
  i = 3 (lính canh) bật j = 2 (a = 3): L = 0, R = 3, số mảng con = 2·1 = 2, đóng góp 6, tổng 7
  i = 3 (lính canh) bật j = 0 (a = 3): L = -1, R = 3, số mảng con = 1·3 = 3, đóng góp 9, tổng 16
Bài 4 ví dụ [3,1,3]: [3, 1, 2]
Bài 1-4: OK


In [22]:
# Bài 5 — tổng range các mảng con độ dài đúng k
from collections import deque
def ranges_fixed_k(a, k):
    dmax, dmin, total = deque(), deque(), 0
    for i, x in enumerate(a):
        while dmax and a[dmax[-1]] <= x: dmax.pop()
        while dmin and a[dmin[-1]] >= x: dmin.pop()
        dmax.append(i); dmin.append(i)
        if dmax[0] <= i - k: dmax.popleft()
        if dmin[0] <= i - k: dmin.popleft()
        if i >= k - 1:
            total += a[dmax[0]] - a[dmin[0]]
    return total

def ranges_fixed_k_brute(a, k):
    return sum(max(a[i:i + k]) - min(a[i:i + k]) for i in range(len(a) - k + 1))

rng = random.Random(55)
for _ in range(1000):
    a = [rng.randint(-5, 5) for _ in range(rng.randint(1, 20))]; k = rng.randint(1, len(a))
    assert ranges_fixed_k(a, k) == ranges_fixed_k_brute(a, k)
assert sum(ranges_fixed_k([4, -2, -3, 4, 1], k) for k in range(1, 6)) == 59
print("Bài 5: khớp brute force; tổng theo mọi k của ví dụ 3 =", sum(ranges_fixed_k([4, -2, -3, 4, 1], k) for k in range(1, 6)))

Bài 5: khớp brute force; tổng theo mọi k của ví dụ 3 = 59


## 20. Template nhớ nhanh + Checklist

```python
# Σ max của mọi mảng con (một phía ngặt, một phía không ngặt) — O(n)
def sum_max(a):
    total, st = 0, []
    for i in range(len(a) + 1):
        cur = a[i] if i < len(a) else float("inf")    # lính canh
        while st and a[st[-1]] < cur:                  # bật khi gặp phần tử LỚN HƠN NGẶT
            j = st.pop()
            L = st[-1] if st else -1                    # phần tử >= a[j] gần nhất bên trái
            total += a[j] * (j - L) * (i - j)
        st.append(i)
    return total

ans = sum_max(nums) + sum_max([-x for x in nums])      # Σmax − Σmin
```

**Checklist trước khi submit:**
- [ ] Một phía so sánh ngặt, phía kia không ngặt (test `[3, 1, 3]`).
- [ ] Lính canh `+∞` cuối mảng (hoặc xử lý phần còn lại).
- [ ] Đúng dấu: `Σmax − Σmin`.
- [ ] Không cắt lát trong vòng lặp `O(n²)`.
- [ ] Kết quả có thể rất lớn (Python ổn; ngôn ngữ khác dùng `long`).